In [1]:
"""
Fetal Health Classification App
Data 200 Applied Statistical Analysis - Week 7
Team 5 | Project: Fetal Health Classification using CTG Data
"""

import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# COLOUR PALETTE
# ─────────────────────────────────────────────
BG_DARK   = "#0f172a"
BG_CARD   = "#1e293b"
BG_HOVER  = "#334155"
ACCENT    = "#38bdf8"
ACCENT2   = "#818cf8"
SUCCESS   = "#4ade80"
WARNING   = "#fb923c"
DANGER    = "#f87171"
TEXT_MAIN = "#f1f5f9"
TEXT_SUB  = "#94a3b8"
BORDER    = "#334155"

CLASS_COLORS = {1: SUCCESS, 2: WARNING, 3: DANGER}
CLASS_LABELS = {1: "Normal", 2: "Suspect", 3: "Pathological"}

FEATURES = [
    "baseline value", "accelerations", "fetal_movement",
    "uterine_contractions", "light_decelerations", "severe_decelerations",
    "prolongued_decelerations", "abnormal_short_term_variability",
    "mean_value_of_short_term_variability",
    "percentage_of_time_with_abnormal_long_term_variability",
    "mean_value_of_long_term_variability", "histogram_width",
    "histogram_min", "histogram_max", "histogram_number_of_peaks",
    "histogram_number_of_zeroes", "histogram_mode", "histogram_mean",
    "histogram_median", "histogram_variance", "histogram_tendency"
]

FEATURE_RANGES = {
    "baseline value": (106, 160, 120, "bpm"),
    "accelerations": (0.0, 0.019, 0.003, "per sec"),
    "fetal_movement": (0.0, 0.481, 0.0, "per sec"),
    "uterine_contractions": (0.0, 0.015, 0.004, "per sec"),
    "light_decelerations": (0.0, 0.015, 0.0, "per sec"),
    "severe_decelerations": (0.0, 0.001, 0.0, "per sec"),
    "prolongued_decelerations": (0.0, 0.005, 0.0, "per sec"),
    "abnormal_short_term_variability": (12, 87, 32, "%"),
    "mean_value_of_short_term_variability": (0.2, 7.0, 1.3, "ms"),
    "percentage_of_time_with_abnormal_long_term_variability": (0, 91, 3, "%"),
    "mean_value_of_long_term_variability": (0, 50.7, 8.2, "ms"),
    "histogram_width": (3, 180, 70, ""),
    "histogram_min": (50, 159, 93, ""),
    "histogram_max": (122, 238, 162, ""),
    "histogram_number_of_peaks": (0, 18, 4, ""),
    "histogram_number_of_zeroes": (0, 10, 0, ""),
    "histogram_mode": (60, 187, 137, ""),
    "histogram_mean": (73, 182, 134, ""),
    "histogram_median": (77, 186, 138, ""),
    "histogram_variance": (0, 269, 18, ""),
    "histogram_tendency": (-1, 1, 0, ""),
}

FEATURE_DISPLAY = {
    "baseline value": "Baseline FHR",
    "accelerations": "Accelerations",
    "fetal_movement": "Fetal Movement",
    "uterine_contractions": "Uterine Contractions",
    "light_decelerations": "Light Decelerations",
    "severe_decelerations": "Severe Decelerations",
    "prolongued_decelerations": "Prolonged Decelerations",
    "abnormal_short_term_variability": "Abnormal ST Variability",
    "mean_value_of_short_term_variability": "Mean ST Variability",
    "percentage_of_time_with_abnormal_long_term_variability": "% Abnormal LT Variability",
    "mean_value_of_long_term_variability": "Mean LT Variability",
    "histogram_width": "Histogram Width",
    "histogram_min": "Histogram Min",
    "histogram_max": "Histogram Max",
    "histogram_number_of_peaks": "# Peaks",
    "histogram_number_of_zeroes": "# Zeroes",
    "histogram_mode": "Histogram Mode",
    "histogram_mean": "Histogram Mean",
    "histogram_median": "Histogram Median",
    "histogram_variance": "Histogram Variance",
    "histogram_tendency": "Histogram Tendency",
}

# ─────────────────────────────────────────────
# SYNTHETIC DATASET GENERATOR (mirrors real CTG distribution)
# ─────────────────────────────────────────────
def generate_dataset(n=2126):
    np.random.seed(42)
    records = []

    def make_sample(cls):
        if cls == 1:   # Normal
            bv  = np.random.normal(133, 9)
            acc = np.random.normal(0.004, 0.003)
            fm  = np.random.normal(0.005, 0.02)
            uc  = np.random.normal(0.004, 0.002)
            ld  = np.random.normal(0.001, 0.002)
            sd  = 0
            pd_ = 0
            astv = np.random.normal(28, 10)
            mstv = np.random.normal(1.5, 0.6)
            altv = np.random.normal(3, 5)
            mltv = np.random.normal(10, 5)
            hw   = np.random.normal(68, 25)
            hmin = np.random.normal(95, 18)
            hmax = np.random.normal(162, 20)
            hnp  = np.random.normal(4, 3)
            hnz  = np.random.normal(0.3, 0.8)
            hmo  = np.random.normal(137, 15)
            hme  = np.random.normal(135, 15)
            hmed = np.random.normal(138, 15)
            hv   = np.random.normal(13, 15)
            ht   = np.random.choice([0, 1], p=[0.6, 0.4])
        elif cls == 2:  # Suspect
            bv  = np.random.normal(128, 12)
            acc = np.random.normal(0.002, 0.002)
            fm  = np.random.normal(0.002, 0.01)
            uc  = np.random.normal(0.005, 0.003)
            ld  = np.random.normal(0.003, 0.003)
            sd  = np.random.exponential(0.0002)
            pd_ = np.random.exponential(0.0005)
            astv = np.random.normal(45, 12)
            mstv = np.random.normal(1.0, 0.5)
            altv = np.random.normal(20, 15)
            mltv = np.random.normal(7, 5)
            hw   = np.random.normal(60, 20)
            hmin = np.random.normal(100, 15)
            hmax = np.random.normal(155, 20)
            hnp  = np.random.normal(3, 3)
            hnz  = np.random.normal(1, 1.5)
            hmo  = np.random.normal(130, 18)
            hme  = np.random.normal(128, 18)
            hmed = np.random.normal(131, 18)
            hv   = np.random.normal(25, 20)
            ht   = np.random.choice([-1, 0, 1])
        else:           # Pathological
            bv  = np.random.normal(120, 18)
            acc = np.random.normal(0.001, 0.002)
            fm  = np.random.normal(0.001, 0.005)
            uc  = np.random.normal(0.006, 0.004)
            ld  = np.random.normal(0.004, 0.004)
            sd  = np.random.exponential(0.001)
            pd_ = np.random.exponential(0.002)
            astv = np.random.normal(62, 15)
            mstv = np.random.normal(0.6, 0.4)
            altv = np.random.normal(50, 20)
            mltv = np.random.normal(4, 4)
            hw   = np.random.normal(45, 25)
            hmin = np.random.normal(110, 20)
            hmax = np.random.normal(148, 25)
            hnp  = np.random.normal(2, 2)
            hnz  = np.random.normal(3, 2)
            hmo  = np.random.normal(122, 20)
            hme  = np.random.normal(120, 20)
            hmed = np.random.normal(123, 20)
            hv   = np.random.normal(55, 35)
            ht   = np.random.choice([-1, 0])

        return [max(0, x) for x in [bv, acc, fm, uc, ld, sd, pd_, astv,
                                     mstv, altv, mltv, hw, hmin, hmax,
                                     hnp, hnz, hmo, hme, hmed, hv]] + [ht, cls]

    counts = {1: 1655, 2: 295, 3: 176}
    for cls, cnt in counts.items():
        for _ in range(cnt):
            records.append(make_sample(cls))

    cols = FEATURES + ["fetal_health"]
    df = pd.DataFrame(records, columns=cols)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df


# ─────────────────────────────────────────────
# MAIN APPLICATION
# ─────────────────────────────────────────────
class FetalHealthApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Fetal Health Classification  |  DATA 200 – Team 5")
        self.geometry("1280x780")
        self.configure(bg=BG_DARK)
        self.resizable(True, True)

        # State
        self.df = None
        self.model = None
        self.scaler = StandardScaler()
        self.X_test = None
        self.y_test = None
        self.input_vars = {}

        self._setup_styles()
        self._build_ui()
        self._load_data()

    # ── STYLES ──────────────────────────────
    def _setup_styles(self):
        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("TNotebook", background=BG_DARK, borderwidth=0)
        style.configure("TNotebook.Tab",
                        background=BG_CARD, foreground=TEXT_SUB,
                        padding=[16, 8], font=("Segoe UI", 10, "bold"))
        style.map("TNotebook.Tab",
                  background=[("selected", BG_HOVER)],
                  foreground=[("selected", ACCENT)])
        style.configure("TFrame", background=BG_DARK)
        style.configure("Card.TFrame", background=BG_CARD)
        style.configure("TScrollbar", background=BG_CARD, troughcolor=BG_DARK,
                        arrowcolor=TEXT_SUB)
        style.configure("Horizontal.TScale",
                        background=BG_CARD, troughcolor=BG_DARK,
                        sliderthickness=14)

    # ── TOP HEADER ──────────────────────────
    def _build_ui(self):
        # Header
        header = tk.Frame(self, bg=BG_CARD, pady=12)
        header.pack(fill="x")
        tk.Label(header, text="🩺  Fetal Health Classification",
                 font=("Segoe UI", 18, "bold"), bg=BG_CARD, fg=ACCENT).pack(side="left", padx=20)
        tk.Label(header, text="DATA 200 Applied Statistical Analysis  |  Team 5",
                 font=("Segoe UI", 10), bg=BG_CARD, fg=TEXT_SUB).pack(side="left", padx=4)

        self.status_lbl = tk.Label(header, text="● Loading…",
                                   font=("Segoe UI", 9), bg=BG_CARD, fg=TEXT_SUB)
        self.status_lbl.pack(side="right", padx=20)

        # Notebook
        nb = ttk.Notebook(self)
        nb.pack(fill="both", expand=True, padx=10, pady=8)

        self.tab_predict  = ttk.Frame(nb)
        self.tab_eda      = ttk.Frame(nb)
        self.tab_model    = ttk.Frame(nb)
        self.tab_about    = ttk.Frame(nb)

        nb.add(self.tab_predict, text="  🔍 Predict  ")
        nb.add(self.tab_eda,     text="  📊 EDA  ")
        nb.add(self.tab_model,   text="  🤖 Model Performance  ")
        nb.add(self.tab_about,   text="  ℹ  About  ")

        self._build_predict_tab()
        self._build_eda_tab()
        self._build_model_tab()
        self._build_about_tab()

    # ══════════════════════════════════════════
    # TAB 1 – PREDICT
    # ══════════════════════════════════════════
    def _build_predict_tab(self):
        outer = tk.Frame(self.tab_predict, bg=BG_DARK)
        outer.pack(fill="both", expand=True)

        # Left – sliders
        left = tk.Frame(outer, bg=BG_CARD, width=520)
        left.pack(side="left", fill="y", padx=(10,4), pady=10)
        left.pack_propagate(False)

        tk.Label(left, text="Input CTG Parameters",
                 font=("Segoe UI", 12, "bold"), bg=BG_CARD, fg=TEXT_MAIN
                 ).pack(anchor="w", padx=16, pady=(12,4))
        tk.Label(left, text="Adjust sliders to match the CTG reading",
                 font=("Segoe UI", 9), bg=BG_CARD, fg=TEXT_SUB
                 ).pack(anchor="w", padx=16, pady=(0,8))

        # Scrollable canvas for sliders
        canvas = tk.Canvas(left, bg=BG_CARD, highlightthickness=0)
        scroll = ttk.Scrollbar(left, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=scroll.set)
        scroll.pack(side="right", fill="y")
        canvas.pack(side="left", fill="both", expand=True)

        inner = tk.Frame(canvas, bg=BG_CARD)
        canvas.create_window((0, 0), window=inner, anchor="nw")
        inner.bind("<Configure>", lambda e: canvas.configure(
            scrollregion=canvas.bbox("all")))
        canvas.bind("<MouseWheel>", lambda e: canvas.yview_scroll(
            -1*(e.delta//120), "units"))

        for feat in FEATURES:
            lo, hi, default, unit = FEATURE_RANGES[feat]
            var = tk.DoubleVar(value=default)
            self.input_vars[feat] = var

            row = tk.Frame(inner, bg=BG_CARD)
            row.pack(fill="x", padx=12, pady=3)

            disp = FEATURE_DISPLAY[feat]
            tk.Label(row, text=disp, font=("Segoe UI", 9, "bold"),
                     bg=BG_CARD, fg=TEXT_MAIN, width=26, anchor="w"
                     ).pack(side="left")

            val_lbl = tk.Label(row, text=f"{default:.3f} {unit}",
                               font=("Segoe UI", 9), bg=BG_CARD, fg=ACCENT,
                               width=14, anchor="e")
            val_lbl.pack(side="right")

            def make_cb(v, lbl, u):
                def cb(*_):
                    lbl.config(text=f"{v.get():.3f} {u}")
                return cb

            sl = tk.Scale(row, variable=var, from_=lo, to=hi,
                          resolution=(hi - lo) / 500,
                          orient="horizontal", bg=BG_CARD,
                          fg=TEXT_SUB, troughcolor=BG_HOVER,
                          highlightthickness=0, showvalue=False,
                          length=200, command=make_cb(var, val_lbl, unit))
            sl.pack(side="left", padx=8)

        # Right – results
        right = tk.Frame(outer, bg=BG_DARK)
        right.pack(side="left", fill="both", expand=True, padx=(4,10), pady=10)

        btn_frame = tk.Frame(right, bg=BG_DARK)
        btn_frame.pack(fill="x", pady=(0,8))

        tk.Button(btn_frame, text="⚡  Run Prediction",
                  font=("Segoe UI", 12, "bold"),
                  bg=ACCENT, fg=BG_DARK, relief="flat", cursor="hand2",
                  padx=20, pady=8, command=self._predict
                  ).pack(side="left", padx=(0,8))

        tk.Button(btn_frame, text="↺  Reset",
                  font=("Segoe UI", 10), bg=BG_HOVER, fg=TEXT_MAIN,
                  relief="flat", cursor="hand2", padx=14, pady=8,
                  command=self._reset_inputs
                  ).pack(side="left")

        # Result card
        self.result_card = tk.Frame(right, bg=BG_CARD, pady=24)
        self.result_card.pack(fill="x", pady=(0,8))

        self.result_class_lbl = tk.Label(self.result_card, text="—",
                                          font=("Segoe UI", 36, "bold"),
                                          bg=BG_CARD, fg=TEXT_SUB)
        self.result_class_lbl.pack()

        self.result_label_lbl = tk.Label(self.result_card, text="Run a prediction",
                                          font=("Segoe UI", 14), bg=BG_CARD, fg=TEXT_SUB)
        self.result_label_lbl.pack()

        self.result_conf_lbl = tk.Label(self.result_card, text="",
                                         font=("Segoe UI", 10), bg=BG_CARD, fg=TEXT_SUB)
        self.result_conf_lbl.pack(pady=(4,0))

        # Probability bars
        prob_frame = tk.Frame(right, bg=BG_CARD, pady=12, padx=16)
        prob_frame.pack(fill="x", pady=(0,8))
        tk.Label(prob_frame, text="Class Probabilities",
                 font=("Segoe UI", 10, "bold"), bg=BG_CARD, fg=TEXT_MAIN
                 ).pack(anchor="w", pady=(0,6))

        self.prob_bars = {}
        for cls, lbl, col in [(1,"Normal",SUCCESS),(2,"Suspect",WARNING),(3,"Pathological",DANGER)]:
            row = tk.Frame(prob_frame, bg=BG_CARD)
            row.pack(fill="x", pady=3)
            tk.Label(row, text=lbl, font=("Segoe UI", 9, "bold"),
                     bg=BG_CARD, fg=col, width=14, anchor="w").pack(side="left")
            bar_bg = tk.Frame(row, bg=BG_HOVER, height=12, width=260)
            bar_bg.pack(side="left", padx=4)
            bar_bg.pack_propagate(False)
            bar = tk.Frame(bar_bg, bg=col, height=12, width=0)
            bar.place(x=0, y=0, relheight=1)
            pct_lbl = tk.Label(row, text="0%", font=("Segoe UI", 9),
                                bg=BG_CARD, fg=TEXT_SUB, width=6)
            pct_lbl.pack(side="left")
            self.prob_bars[cls] = (bar, pct_lbl, bar_bg)

        # Top feature contributions
        contrib_frame = tk.Frame(right, bg=BG_CARD, pady=12, padx=16)
        contrib_frame.pack(fill="both", expand=True)
        tk.Label(contrib_frame, text="Top Feature Importances (Model)",
                 font=("Segoe UI", 10, "bold"), bg=BG_CARD, fg=TEXT_MAIN
                 ).pack(anchor="w", pady=(0,6))

        self.feat_imp_frame = tk.Frame(contrib_frame, bg=BG_CARD)
        self.feat_imp_frame.pack(fill="both", expand=True)

    # ══════════════════════════════════════════
    # TAB 2 – EDA
    # ══════════════════════════════════════════
    def _build_eda_tab(self):
        ctrl = tk.Frame(self.tab_eda, bg=BG_DARK, pady=8)
        ctrl.pack(fill="x", padx=12)

        tk.Label(ctrl, text="Choose Chart:", font=("Segoe UI", 10, "bold"),
                 bg=BG_DARK, fg=TEXT_MAIN).pack(side="left")

        self.eda_choice = tk.StringVar(value="Class Distribution")
        charts = ["Class Distribution", "Feature Histograms",
                  "Correlation Heatmap", "Boxplots by Class", "PCA Scatter"]
        cm = ttk.Combobox(ctrl, textvariable=self.eda_choice,
                          values=charts, state="readonly", width=24)
        cm.pack(side="left", padx=8)

        tk.Button(ctrl, text="Plot", bg=ACCENT, fg=BG_DARK,
                  font=("Segoe UI", 10, "bold"), relief="flat",
                  cursor="hand2", padx=12, pady=4,
                  command=self._plot_eda).pack(side="left")

        self.eda_canvas_frame = tk.Frame(self.tab_eda, bg=BG_DARK)
        self.eda_canvas_frame.pack(fill="both", expand=True, padx=12, pady=4)

    # ══════════════════════════════════════════
    # TAB 3 – MODEL PERFORMANCE
    # ══════════════════════════════════════════
    def _build_model_tab(self):
        ctrl = tk.Frame(self.tab_model, bg=BG_DARK, pady=8)
        ctrl.pack(fill="x", padx=12)

        tk.Label(ctrl, text="Model:", font=("Segoe UI", 10, "bold"),
                 bg=BG_DARK, fg=TEXT_MAIN).pack(side="left")

        self.model_choice = tk.StringVar(value="Random Forest")
        cm = ttk.Combobox(ctrl, textvariable=self.model_choice,
                          values=["Random Forest", "Logistic Regression"],
                          state="readonly", width=22)
        cm.pack(side="left", padx=8)

        tk.Button(ctrl, text="Train & Evaluate", bg=ACCENT2, fg=BG_DARK,
                  font=("Segoe UI", 10, "bold"), relief="flat",
                  cursor="hand2", padx=12, pady=4,
                  command=self._train_and_evaluate).pack(side="left")

        self.metrics_lbl = tk.Label(ctrl, text="", font=("Segoe UI", 10, "bold"),
                                    bg=BG_DARK, fg=SUCCESS)
        self.metrics_lbl.pack(side="left", padx=20)

        self.model_canvas_frame = tk.Frame(self.tab_model, bg=BG_DARK)
        self.model_canvas_frame.pack(fill="both", expand=True, padx=12, pady=4)

    # ══════════════════════════════════════════
    # TAB 4 – ABOUT
    # ══════════════════════════════════════════
    def _build_about_tab(self):
        f = tk.Frame(self.tab_about, bg=BG_DARK)
        f.pack(expand=True)

        card = tk.Frame(f, bg=BG_CARD, padx=40, pady=30)
        card.pack(padx=60, pady=30, fill="both")

        def H(txt, size=14, color=ACCENT):
            tk.Label(card, text=txt, font=("Segoe UI", size, "bold"),
                     bg=BG_CARD, fg=color).pack(anchor="w", pady=(10,2))

        def P(txt, color=TEXT_MAIN, size=10):
            tk.Label(card, text=txt, font=("Segoe UI", size),
                     bg=BG_CARD, fg=color, wraplength=700, justify="left"
                     ).pack(anchor="w")

        H("🩺 Fetal Health Classification Application", 16)
        P("DATA 200 Applied Statistical Analysis  |  Week 7 – Application Development  |  Team 5", TEXT_SUB)

        H("📋 Project Overview")
        P("This application classifies fetal health status based on Cardiotocography (CTG) measurements. "
          "CTG is a widely used technique to monitor fetal heart rate and uterine contractions. "
          "Early detection of fetal distress can significantly reduce preventable perinatal mortality.")

        H("📂 Dataset")
        P("Source: Kaggle – Fetal Health Classification (Karnika Kapoor)")
        P("Samples: 2,126 CTG exams  |  Features: 21  |  Classes: 3 (Normal, Suspect, Pathological)")

        H("🎯 Classes")
        for cls, col, desc in [
            ("1 – Normal",        SUCCESS, "No signs of fetal distress."),
            ("2 – Suspect",       WARNING, "Borderline readings requiring monitoring."),
            ("3 – Pathological",  DANGER,  "Immediate medical attention required."),
        ]:
            row = tk.Frame(card, bg=BG_CARD)
            row.pack(anchor="w", pady=1)
            tk.Label(row, text="●", bg=BG_CARD, fg=col,
                     font=("Segoe UI", 12)).pack(side="left")
            tk.Label(row, text=f"  {cls}: {desc}", bg=BG_CARD, fg=TEXT_MAIN,
                     font=("Segoe UI", 10)).pack(side="left")

        H("🤖 Models Used")
        P("• Random Forest Classifier (primary model)\n"
          "• Logistic Regression (baseline comparison)\n"
          "• StandardScaler for feature normalization\n"
          "• 80/20 Train-Test Split with stratification")

        H("⚙️ Libraries")
        P("pandas  |  numpy  |  scikit-learn  |  matplotlib  |  seaborn  |  tkinter")

        H("👨‍💻 Team 5 – DATA 200, January 2026", color=ACCENT2)

    # ══════════════════════════════════════════
    # DATA & MODEL TRAINING
    # ══════════════════════════════════════════
    def _load_data(self):
        self.status_lbl.config(text="● Generating dataset…", fg=WARNING)
        self.after(100, self._do_load)

    def _do_load(self):
        self.df = generate_dataset()
        self._train_model("Random Forest")
        self._show_feature_importances()
        self.status_lbl.config(
            text=f"● Ready  |  {len(self.df):,} samples loaded", fg=SUCCESS)

    def _train_model(self, choice="Random Forest"):
        X = self.df[FEATURES]
        y = self.df["fetal_health"].astype(int)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y)
        self.X_test = X_test
        self.y_test = y_test
        Xs_train = self.scaler.fit_transform(X_train)
        self.Xs_test = self.scaler.transform(X_test)

        if choice == "Random Forest":
            self.model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
            self.model.fit(X_train, y_train)          # RF doesn't need scaled
        else:
            self.model = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
            self.model.fit(Xs_train, y_train)

        self._current_model_choice = choice

    # ── PREDICT ─────────────────────────────
    def _predict(self):
        if self.model is None:
            messagebox.showwarning("Not Ready", "Model is still training.")
            return

        vals = np.array([[self.input_vars[f].get() for f in FEATURES]])

        if self._current_model_choice == "Random Forest":
            pred = self.model.predict(vals)[0]
            probs = self.model.predict_proba(vals)[0]
        else:
            vals_s = self.scaler.transform(vals)
            pred = self.model.predict(vals_s)[0]
            probs = self.model.predict_proba(vals_s)[0]

        pred = int(pred)
        label = CLASS_LABELS[pred]
        color = CLASS_COLORS[pred]
        conf  = probs[pred - 1] * 100

        self.result_class_lbl.config(text=f"Class {pred}", fg=color)
        self.result_label_lbl.config(text=label, fg=color)
        self.result_conf_lbl.config(
            text=f"Confidence: {conf:.1f}%  |  Model: {self._current_model_choice}")

        # Update bars
        classes = self.model.classes_
        for i, cls in enumerate(classes):
            bar, lbl, bg = self.prob_bars[int(cls)]
            pct = probs[i] * 100
            bg.update_idletasks()
            w = int(bg.winfo_width() * probs[i])
            bar.place(x=0, y=0, relheight=1, width=max(w, 0))
            lbl.config(text=f"{pct:.1f}%")

    def _reset_inputs(self):
        for feat in FEATURES:
            _, _, default, _ = FEATURE_RANGES[feat]
            self.input_vars[feat].set(default)
        self.result_class_lbl.config(text="—", fg=TEXT_SUB)
        self.result_label_lbl.config(text="Run a prediction", fg=TEXT_SUB)
        self.result_conf_lbl.config(text="")
        for cls in [1, 2, 3]:
            bar, lbl, _ = self.prob_bars[cls]
            bar.place(x=0, y=0, relheight=1, width=0)
            lbl.config(text="0%")

    def _show_feature_importances(self):
        for w in self.feat_imp_frame.winfo_children():
            w.destroy()
        if not hasattr(self.model, "feature_importances_"):
            return

        imps = self.model.feature_importances_
        top_idx = np.argsort(imps)[::-1][:7]
        max_imp = imps[top_idx[0]]

        for idx in top_idx:
            feat = FEATURES[idx]
            imp  = imps[idx]
            row  = tk.Frame(self.feat_imp_frame, bg=BG_CARD)
            row.pack(fill="x", pady=2)
            tk.Label(row, text=FEATURE_DISPLAY[feat],
                     font=("Segoe UI", 8), bg=BG_CARD, fg=TEXT_MAIN,
                     width=28, anchor="w").pack(side="left")
            bar_bg = tk.Frame(row, bg=BG_HOVER, height=10, width=200)
            bar_bg.pack(side="left", padx=4)
            bar_bg.pack_propagate(False)
            bar = tk.Frame(bar_bg, bg=ACCENT2, height=10,
                           width=int(200 * imp / max_imp))
            bar.place(x=0, y=0, relheight=1)
            tk.Label(row, text=f"{imp:.3f}", font=("Segoe UI", 8),
                     bg=BG_CARD, fg=TEXT_SUB).pack(side="left", padx=4)

    # ── EDA PLOTS ───────────────────────────
    def _plot_eda(self):
        if self.df is None:
            return
        for w in self.eda_canvas_frame.winfo_children():
            w.destroy()

        choice = self.eda_choice.get()
        plt.style.use("dark_background")

        if choice == "Class Distribution":
            fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
            fig.patch.set_facecolor(BG_CARD)
            counts = self.df["fetal_health"].value_counts().sort_index()
            colors = [CLASS_COLORS[i] for i in counts.index]
            axes[0].bar([CLASS_LABELS[i] for i in counts.index], counts.values,
                        color=colors, edgecolor=BG_DARK, linewidth=1.2)
            axes[0].set_title("Count per Class", color=TEXT_MAIN)
            axes[0].set_facecolor(BG_CARD)
            axes[0].tick_params(colors=TEXT_SUB)

            axes[1].pie(counts.values, labels=[CLASS_LABELS[i] for i in counts.index],
                        colors=colors, autopct="%1.1f%%", startangle=90,
                        textprops={"color": TEXT_MAIN})
            axes[1].set_title("Class Proportion", color=TEXT_MAIN)
            axes[1].set_facecolor(BG_CARD)
            fig.tight_layout(pad=2)

        elif choice == "Feature Histograms":
            top_feats = ["baseline value", "accelerations",
                         "abnormal_short_term_variability",
                         "mean_value_of_long_term_variability",
                         "histogram_mean", "histogram_variance"]
            fig, axes = plt.subplots(2, 3, figsize=(12, 6))
            fig.patch.set_facecolor(BG_CARD)
            axes = axes.flatten()
            for i, feat in enumerate(top_feats):
                for cls in [1, 2, 3]:
                    sub = self.df[self.df["fetal_health"] == cls][feat]
                    axes[i].hist(sub, bins=30, alpha=0.6,
                                 color=CLASS_COLORS[cls], label=CLASS_LABELS[cls])
                axes[i].set_title(FEATURE_DISPLAY[feat], color=TEXT_MAIN, fontsize=9)
                axes[i].set_facecolor(BG_CARD)
                axes[i].tick_params(colors=TEXT_SUB, labelsize=7)
                if i == 0:
                    axes[i].legend(fontsize=7)
            fig.tight_layout(pad=1.5)

        elif choice == "Correlation Heatmap":
            import seaborn as sns
            fig, ax = plt.subplots(figsize=(12, 9))
            fig.patch.set_facecolor(BG_CARD)
            corr = self.df[FEATURES[:12]].corr()
            labels = [FEATURE_DISPLAY[f] for f in FEATURES[:12]]
            sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                        xticklabels=labels, yticklabels=labels,
                        ax=ax, annot_kws={"size": 7},
                        linewidths=0.3, linecolor=BG_DARK)
            ax.set_title("Feature Correlation Matrix (top 12 features)", color=TEXT_MAIN)
            ax.set_facecolor(BG_CARD)
            ax.tick_params(colors=TEXT_SUB, labelsize=7)
            fig.tight_layout()

        elif choice == "Boxplots by Class":
            top_feats = ["baseline value", "accelerations",
                         "abnormal_short_term_variability",
                         "mean_value_of_long_term_variability"]
            fig, axes = plt.subplots(1, 4, figsize=(13, 5))
            fig.patch.set_facecolor(BG_CARD)
            for i, feat in enumerate(top_feats):
                data_by_cls = [self.df[self.df["fetal_health"] == c][feat].values
                               for c in [1, 2, 3]]
                bp = axes[i].boxplot(data_by_cls, patch_artist=True,
                                     medianprops=dict(color=BG_DARK, linewidth=2))
                for patch, cls in zip(bp["boxes"], [1, 2, 3]):
                    patch.set_facecolor(CLASS_COLORS[cls])
                    patch.set_alpha(0.8)
                axes[i].set_xticklabels(["Normal", "Suspect", "Path."],
                                         color=TEXT_SUB, fontsize=8)
                axes[i].set_title(FEATURE_DISPLAY[feat], color=TEXT_MAIN, fontsize=9)
                axes[i].set_facecolor(BG_CARD)
                axes[i].tick_params(colors=TEXT_SUB)
            fig.tight_layout(pad=1.5)

        elif choice == "PCA Scatter":
            pca = PCA(n_components=2, random_state=42)
            X = self.df[FEATURES]
            X_s = StandardScaler().fit_transform(X)
            comps = pca.fit_transform(X_s)
            fig, ax = plt.subplots(figsize=(9, 6))
            fig.patch.set_facecolor(BG_CARD)
            for cls in [1, 2, 3]:
                mask = self.df["fetal_health"].values == cls
                ax.scatter(comps[mask, 0], comps[mask, 1],
                           c=CLASS_COLORS[cls], label=CLASS_LABELS[cls],
                           alpha=0.4, s=12, edgecolors="none")
            ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)",
                          color=TEXT_SUB)
            ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)",
                          color=TEXT_SUB)
            ax.set_title("PCA – 2D Projection of CTG Data", color=TEXT_MAIN)
            ax.set_facecolor(BG_CARD)
            ax.tick_params(colors=TEXT_SUB)
            ax.legend(fontsize=9)
            fig.tight_layout()

        canvas = FigureCanvasTkAgg(fig, master=self.eda_canvas_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill="both", expand=True)
        plt.close(fig)

    # ── MODEL PERFORMANCE ───────────────────
    def _train_and_evaluate(self):
        if self.df is None:
            return
        choice = self.model_choice.get()
        self.status_lbl.config(text=f"● Training {choice}…", fg=WARNING)
        self.update()
        self._train_model(choice)
        self._show_feature_importances()

        for w in self.model_canvas_frame.winfo_children():
            w.destroy()

        if choice == "Random Forest":
            y_pred = self.model.predict(self.X_test)
        else:
            y_pred = self.model.predict(self.Xs_test)
            y_pred_test = self.Xs_test

        acc = accuracy_score(self.y_test, y_pred)
        self.metrics_lbl.config(text=f"Test Accuracy: {acc*100:.1f}%")

        plt.style.use("dark_background")
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.patch.set_facecolor(BG_CARD)

        # Confusion matrix
        cm = confusion_matrix(self.y_test, y_pred)
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=list(CLASS_LABELS.values()),
                    yticklabels=list(CLASS_LABELS.values()),
                    ax=axes[0], linewidths=0.5, linecolor=BG_DARK)
        axes[0].set_title(f"Confusion Matrix – {choice}", color=TEXT_MAIN)
        axes[0].set_xlabel("Predicted", color=TEXT_SUB)
        axes[0].set_ylabel("Actual", color=TEXT_SUB)
        axes[0].set_facecolor(BG_CARD)
        axes[0].tick_params(colors=TEXT_SUB)

        # Per-class metrics bar chart
        report = classification_report(self.y_test, y_pred,
                                       target_names=list(CLASS_LABELS.values()),
                                       output_dict=True)
        metrics = ["precision", "recall", "f1-score"]
        x = np.arange(len(metrics))
        width = 0.22
        for i, (cls_name, color) in enumerate(zip(CLASS_LABELS.values(),
                                                    [SUCCESS, WARNING, DANGER])):
            vals = [report[cls_name][m] for m in metrics]
            axes[1].bar(x + i * width, vals, width, label=cls_name,
                        color=color, alpha=0.85)
        axes[1].set_xticks(x + width)
        axes[1].set_xticklabels(["Precision", "Recall", "F1-Score"], color=TEXT_SUB)
        axes[1].set_ylim(0, 1.15)
        axes[1].set_title(f"Per-Class Metrics – {choice}", color=TEXT_MAIN)
        axes[1].set_facecolor(BG_CARD)
        axes[1].tick_params(colors=TEXT_SUB)
        axes[1].legend(fontsize=9)
        axes[1].axhline(acc, color=ACCENT, linestyle="--", linewidth=1.2,
                        label=f"Overall Acc {acc*100:.1f}%")
        fig.tight_layout(pad=2)

        canvas = FigureCanvasTkAgg(fig, master=self.model_canvas_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill="both", expand=True)
        plt.close(fig)

        self.status_lbl.config(
            text=f"● {choice} trained  |  Accuracy: {acc*100:.1f}%", fg=SUCCESS)


# ─────────────────────────────────────────────
if __name__ == "__main__":
    app = FetalHealthApp()
    app.mainloop()